# BERT LoRA 对照实验（LORA_Funning）

## 目的
本Notebook使用 `Bert_Config.py` 的统一配置与通用函数，进行BERT模型的LoRA（Low-Rank Adaptation）微调训练与评估实验。

## 数据流向
输入：原始数据文件（waimai.csv）-> 数据预处理 -> LoRA微调训练 -> 模型评估
输出：训练好的LoRA适配器、训练过程指标、测试集评估结果

## 操作步骤
1. 导入必要的库和配置（包括PEFT库）
2. 加载和预处理数据
3. 初始化BERT模型并应用LoRA适配器
4. 配置训练环境和优化器
5. 执行训练循环，包含验证和早停机制
6. 在测试集上评估模型性能

In [2]:
"""
目的：导入实验所需的所有Python库和模块，包括PEFT库用于LoRA微调

数据流向：
输入：无（直接导入库）
输出：所有必要的库和函数已加载到当前命名空间

操作步骤：
1. 导入系统库：os（文件操作）、time（时间计算）
2. 导入PyTorch相关：torch（张量计算）、nn（神经网络模块）、DataLoader（数据加载）
3. 导入评估指标：sklearn的准确率、精确率、召回率、F1分数计算函数
4. 导入Transformers库：BERT配置、模型、学习率调度器
5. 导入优化器：AdamW（带权重衰减的Adam优化器）
6. 导入进度条：tqdm（显示训练进度）
7. 导入PEFT库：LoraConfig（LoRA配置）、get_peft_model（应用LoRA）、TaskType（任务类型）
8. 导入自定义配置：从Bert_Config导入统一配置和工具函数
"""

import os
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertConfig, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

from peft import LoraConfig, get_peft_model, TaskType

from Bert_Config import CONFIG, setup_seed, build_tokenizer, generate_data, get_save_path

print("✅ 所有库导入完成")

C:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成


In [3]:
"""
目的：从统一配置文件加载实验参数，设置LoRA微调实验的特定配置，包括LoRA相关参数

数据流向：
输入：CONFIG字典（从Bert_Config导入）-> 提取配置参数 -> 设置LoRA特定参数
输出：所有配置变量已设置，模型路径已验证，配置信息已打印

操作步骤：
1. 从CONFIG字典提取通用配置：模型路径、数据路径、训练轮数、批次大小、最大长度、dropout、类别数、权重衰减、随机种子
2. 设置LoRA微调特定参数：学习率（1e-4）、FREEZE_BERT=False（不冻结BERT参数）
3. 设置LoRA特定参数：
   - LORA_R=8：LoRA的秩（低秩矩阵的维度）
   - LORA_ALPHA=16：LoRA的缩放系数（控制适配器的影响强度）
   - LORA_DROPOUT=0.1：LoRA层的dropout率
   - LORA_TARGET_MODULES：应用LoRA的目标模块（query、value、key注意力层）
4. 获取模型保存路径：调用get_save_path("lora")获取保存目录
5. 验证BERT模型路径是否存在
6. 设置实验名称：EXP_NAME = "LoRA微调"
7. 打印所有配置参数，包括LoRA相关参数
"""

MODEL_PATH = CONFIG["BERT_MODEL_PATH"]
DATA_PATH = CONFIG["DATA_DIR"]
EPOCHS = CONFIG["EPOCHS"]
LEARNING_RATE = 1e-4
BATCH_SIZE = CONFIG["BATCH_SIZE"]
MAX_LENGTH = CONFIG["MAX_LENGTH"]
DROPOUT = CONFIG["DROPOUT"]
NUM_CLASSES = CONFIG["NUM_CLASSES"]
WEIGHT_DECAY = CONFIG["WEIGHT_DECAY"]
FREEZE_BERT = False
RANDOM_SEED = int(os.environ.get("NOTEBOOK_SEED", CONFIG["RANDOM_SEED"]))
SAVE_PATH = get_save_path("lora")

LORA_R = CONFIG["LORA_R"]
LORA_ALPHA = CONFIG["LORA_ALPHA"]
LORA_DROPOUT = CONFIG["LORA_DROPOUT"]
LORA_TARGET_MODULES = CONFIG["LORA_TARGET_MODULES"]

if not os.path.exists(MODEL_PATH):
    print(f"⚠️  模型文件夹不存在: {MODEL_PATH}")
else:
    print(f"✅ 模型路径检查通过: {MODEL_PATH}")

EXP_NAME = "LoRA微调"
print("=" * 50)
print(f"实验配置: {EXP_NAME}")
print("=" * 50)
print(f"BERT模型路径: {MODEL_PATH}")
print(f"数据目录: {DATA_PATH}")
print(f"训练轮数: {EPOCHS}")
print(f"学习率: {LEARNING_RATE}")
print(f"批次大小: {BATCH_SIZE}")
print(f"最大文本长度: {MAX_LENGTH}")
print(f"dropout率 (DROPOUT): {DROPOUT}")
print(f"分类类别数: {NUM_CLASSES}")
print(f"权重衰减: {WEIGHT_DECAY}")
print(f"是否冻结BERT (FREEZE_BERT): {FREEZE_BERT}")
print(f"随机种子: {RANDOM_SEED}")
print(f"模型保存路径: {SAVE_PATH}")
print(f"LoRA秩: {LORA_R}")
print(f"LoRA缩放系数: {LORA_ALPHA}")
print(f"LoRA dropout率: {LORA_DROPOUT}")
print(f"LoRA目标模块: {LORA_TARGET_MODULES}")
print("=" * 50)

print("✅ 配置加载完成")

✅ 模型路径检查通过: ../bert-base-chinese
实验配置: LoRA微调
BERT模型路径: ../bert-base-chinese
数据目录: ../waimai.csv
训练轮数: 5
学习率: 0.0003
批次大小: 32
最大文本长度: 256
dropout率 (DROPOUT): 0.1
分类类别数: 2
权重衰减: 0.01
是否冻结BERT (FREEZE_BERT): False
随机种子: 42
模型保存路径: ./bert_lora_checkpoint
LoRA秩: 8
LoRA缩放系数: 16
LoRA dropout率: 0.1
LoRA目标模块: ['query', 'value']
✅ 配置加载完成


In [ ]:
"""
目的：设置随机种子，确保实验的可重复性，使每次运行结果一致

数据流向：
输入：RANDOM_SEED（随机种子值）-> setup_seed函数 -> 设置Python、NumPy、PyTorch的随机种子
输出：所有随机数生成器的种子已设置，实验结果可复现

操作步骤：
1. 调用setup_seed函数，传入RANDOM_SEED参数
2. setup_seed内部设置：Python的random模块、NumPy的随机数生成器、PyTorch的随机数生成器（CPU和CUDA）
3. 打印设置完成提示
"""

setup_seed(RANDOM_SEED)
print("✅ 随机种子设置完成")

In [ ]:
"""
目的：加载BERT模型对应的中文分词器，用于将文本转换为模型可处理的token ID序列

数据流向：
输入：MODEL_PATH（BERT模型路径）-> build_tokenizer函数 -> 加载分词器
输出：tokenizer对象（包含词汇表和编码/解码方法）

操作步骤：
1. 调用build_tokenizer函数，传入BERT模型路径
2. build_tokenizer从模型目录加载预训练的分词器（通常是BertTokenizer）
3. 分词器包含词汇表，可以将中文文本转换为token ID序列
4. 打印分词器加载成功信息和词汇表大小
"""

tokenizer = build_tokenizer(MODEL_PATH)
print("✅ 分词器加载完成")
print(f"   - 词汇表大小: {len(tokenizer.vocab)}")

In [ ]:
"""
目的：加载原始数据，进行预处理和分词，创建训练集和验证集的数据集对象和数据加载器

数据流向：
输入：DATA_PATH（数据文件路径）、tokenizer（分词器）、MAX_LENGTH（最大长度）、RANDOM_SEED（随机种子）
-> generate_data函数处理数据 -> 数据集对象 -> DataLoader封装
输出：train_dataloader（训练数据加载器）、val_dataloader（验证数据加载器）

操作步骤：
1. 调用generate_data生成训练集：传入"train"、数据路径、分词器、最大长度、随机种子
   - generate_data内部：读取CSV文件、按8:1:1划分数据、对文本进行分词和编码、创建PyTorch数据集
2. 调用generate_data生成验证集：传入"val"参数，使用相同的数据划分种子确保一致性
3. 创建训练数据加载器：使用DataLoader封装训练集，设置批次大小和shuffle=True（打乱数据）
4. 创建验证数据加载器：使用DataLoader封装验证集，设置批次大小，不shuffle
5. 打印数据准备完成信息和批次数量
"""

train_dataset = generate_data("train", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)
val_dataset = generate_data("val", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print("✅ 训练数据准备完成")
print(f"   - 训练批次数: {len(train_dataloader)}")
print(f"   - 验证批次数: {len(val_dataloader)}")

In [ ]:
"""
目的：加载预训练的BERT模型，配置为序列分类任务，然后应用LoRA适配器进行参数高效微调

数据流向：
输入：MODEL_PATH（模型路径）、DROPOUT（dropout率）、NUM_CLASSES（类别数）、LoRA配置参数
-> 加载BERT配置和模型 -> 创建LoRA配置 -> 应用LoRA适配器
输出：model（应用了LoRA的BERT模型对象）

操作步骤：
1. 从预训练模型路径加载BERT配置：BertConfig.from_pretrained(MODEL_PATH)
2. 修改配置参数：hidden_dropout_prob、attention_probs_dropout_prob、num_labels
3. 加载BERT序列分类模型：BertForSequenceClassification.from_pretrained
4. 如果FREEZE_BERT为True，冻结BERT主干参数（本实验为False）
5. 打印原始模型参数数量
6. 创建LoRA配置对象：
   - task_type=TaskType.SEQ_CLS（序列分类任务）
   - r=LORA_R（LoRA秩）
   - lora_alpha=LORA_ALPHA（缩放系数）
   - lora_dropout=LORA_DROPOUT（dropout率）
   - target_modules=LORA_TARGET_MODULES（目标模块：query、value、key）
   - bias="none"（不训练bias）
   - inference_mode=False（训练模式）
7. 使用get_peft_model将LoRA适配器应用到模型
8. 打印可训练参数信息（LoRA参数数量远少于原始参数）
9. 计算并打印参数效率（可训练参数占总参数的比例）
"""

config = BertConfig.from_pretrained(MODEL_PATH)
config.hidden_dropout_prob = DROPOUT
config.attention_probs_dropout_prob = DROPOUT
config.num_labels=NUM_CLASSES

model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=config,
)
if FREEZE_BERT:
    for param in model.bert.parameters():
        param.requires_grad = False
    print("bert主干参数已经被冻结")

print("✅ BERT基础模型加载完成")
print(f"   - 原始模型参数数量: {sum(p.numel() for p in model.parameters()):,}")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA配置完成")
print(f"   - LoRA参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(
    f"   - 参数效率: {sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters()) * 100:.2f}%"
)

In [ ]:
"""
目的：配置训练所需的环境，包括计算设备、损失函数、优化器和学习率调度器

数据流向：
输入：model（模型）、LEARNING_RATE（学习率）、WEIGHT_DECAY（权重衰减）、EPOCHS（训练轮数）、train_dataloader（训练数据）
-> 检测设备 -> 创建损失函数 -> 创建优化器 -> 创建学习率调度器 -> 移动模型到设备
输出：device（计算设备）、criterion（损失函数）、optimizer（优化器）、scheduler（学习率调度器）

操作步骤：
1. 检测CUDA是否可用：torch.cuda.is_available()
2. 设置计算设备：如果有CUDA则使用GPU，否则使用CPU
3. 创建交叉熵损失函数：nn.CrossEntropyLoss（适用于多分类任务）
4. 创建AdamW优化器：
   - 传入模型所有参数（LoRA模型只包含可训练参数）
   - 设置学习率、权重衰减、数值稳定性参数eps
5. 计算总训练步数：训练批次数 × 训练轮数
6. 计算预热步数：总步数的10%（用于学习率预热）
7. 创建线性学习率调度器（带预热）：
   - 前warmup_steps步线性增加学习率
   - 后续步数线性减少学习率
8. 如果使用CUDA，将模型和损失函数移动到GPU
9. 打印配置完成信息和训练步数信息
"""

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"使用设备: {device}")

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eps=1e-8,
)

total_steps = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

if use_cuda:
    model = model.cuda()
    criterion = criterion.cuda()

print("✅ 训练环境配置完成")
print(f"   - 总训练步数: {total_steps}")
print(f"   - 预热步数: {warmup_steps}")

In [ ]:
"""
目的：定义模型评估函数、早停机制类和LoRA模型保存函数，用于训练过程中的验证和模型管理

数据流向：
输入：model（模型）、dataloader（数据加载器）、criterion（损失函数）、device（设备）
-> 批量评估 -> 收集预测和标签 -> 计算指标
输出：评估指标元组（准确率、精确率、召回率、F1分数、平均损失）

操作步骤：
1. 定义evaluate_bert函数：评估BERT模型性能（适配LoRA模型）
2. 定义EarlyStopping类：实现早停机制，防止过拟合
3. 定义save_lora_model函数：保存LoRA适配器参数到磁盘
"""

def evaluate_bert(model, dataloader, criterion, device):
    """
    目的：评估BERT模型（包括LoRA模型）在指定数据集上的性能，计算多项分类指标
    
    输入：
      - model: 训练好的BERT模型（可能是LoRA模型）
      - dataloader: 数据加载器（验证集或测试集）
      - criterion: 损失函数（交叉熵损失）
      - device: 计算设备（CPU或GPU）
    输出：
      - acc: 准确率（float）
      - precision: 精确率（float）
      - recall: 召回率（float）
      - f1: F1分数（float）
      - avg_loss: 平均损失（float）
    
    数据流向：
    输入数据批次 -> 模型前向传播（传入labels）-> 计算损失和预测 -> 收集所有结果 -> 计算指标
    
    操作步骤：
    1. 将模型设置为评估模式（关闭dropout等训练行为）
    2. 初始化累计损失和标签/预测列表
    3. 关闭梯度计算（节省内存）
    4. 遍历数据加载器中的每个批次：
       a. 将标签、attention_mask、input_ids移动到指定设备
       b. 通过模型前向传播得到logits（LoRA模型可以传入labels，但评估时使用criterion计算损失以保持一致性）
       c. 使用criterion计算批次损失并累加（乘以批次大小）
       d. 通过argmax获取预测类别
       e. 收集真实标签和预测标签到列表
    5. 计算平均损失（总损失除以样本总数）
    6. 使用sklearn计算准确率、精确率、召回率、F1分数（二分类）
    7. 返回所有评估指标
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            labels = labels.to(device)
            mask = inputs["attention_mask"].to(device)
            input_ids = inputs["input_ids"].squeeze(1).to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)
            logits = outputs.logits
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            
            preds = logits.argmax(dim=1)
            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
    
    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    return acc, precision, recall, f1, avg_loss


class EarlyStopping:
    """
    目的：实现早停机制，当验证指标连续多个epoch不再提升时停止训练，防止过拟合
    
    输入：
      - patience: 容忍的连续不提升epoch数（默认3）
      - min_delta: 最小提升阈值（默认0.001）
    输出：
      - __call__方法返回True表示应该早停，False表示继续训练
    
    数据流向：
    验证指标 -> 与历史最佳比较 -> 判断是否提升 -> 更新计数器 -> 返回是否早停
    
    操作步骤：
    1. 初始化：设置patience、min_delta、计数器、最佳分数
    2. __call__方法：
       a. 如果是第一次调用，记录当前分数为最佳分数
       b. 如果当前分数 < 最佳分数 + min_delta（未提升），计数器+1
       c. 如果计数器 >= patience，返回True（触发早停）
       d. 如果当前分数 >= 最佳分数 + min_delta（有提升），更新最佳分数，重置计数器
       e. 返回False（继续训练）
    """
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        
    def __call__(self, val_score):
        if self.best_score is None:
            self.best_score = val_score
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        else:
            self.best_score = val_score
            self.counter = 0
        return False


def save_lora_model(model, save_name):
    """
    目的：将训练好的LoRA适配器参数保存到磁盘，使用PEFT的save_pretrained方法
    
    输入：
      - model: 应用了LoRA的模型对象
      - save_name: 保存的目录名（如"best"或"last"）
    输出：无（LoRA适配器参数保存到目录）
    
    数据流向：
    LoRA适配器参数 -> save_pretrained方法 -> 保存到目录（包含adapter_config.json和adapter_model.bin）
    
    操作步骤：
    1. 拼接完整的保存路径（SAVE_PATH + save_name）
    2. 调用模型的save_pretrained方法，保存LoRA适配器配置和参数
    3. 打印保存成功的消息和目录路径
    """
    save_dir = os.path.join(SAVE_PATH, save_name)
    model.save_pretrained(save_dir)
    print(f"✅ LoRA模型已保存: {save_dir}")

print("✅ 评估函数和早停机制定义完成")

In [ ]:
"""
目的：执行LoRA模型训练循环，包括前向传播、反向传播、参数更新、验证评估和模型保存

数据流向：
输入：train_dataloader（训练数据）、val_dataloader（验证数据）、model（LoRA模型）、optimizer（优化器）
-> 训练循环 -> 每个epoch：训练批次 -> 验证评估 -> 保存最佳模型
输出：训练好的LoRA适配器（保存为best和last目录）、训练指标、训练时间

操作步骤：
1. 初始化训练开始时间、最佳验证准确率、早停机制对象
2. 对每个epoch执行：
   a. 设置模型为训练模式
   b. 初始化训练准确率和损失累计器
   c. 遍历训练数据批次：
      - 将数据移动到设备
      - 前向传播得到logits和loss（LoRA模型可以直接返回loss）
      - 反向传播计算梯度
      - 梯度裁剪（防止梯度爆炸）
      - 优化器更新参数（只更新LoRA参数）
      - 学习率调度器更新学习率
   d. 在验证集上评估模型性能
   e. 计算训练集平均损失和准确率
   f. 打印epoch训练和验证指标
   g. 如果验证准确率提升，保存最佳LoRA适配器
   h. 检查早停条件，如果触发则提前结束训练
3. 训练结束后保存最后一个epoch的LoRA适配器
4. 打印训练完成信息和模型保存路径
"""

print("\n开始训练...")
train_start_time = time.time()
best_dev_acc = 0
early_stopping = EarlyStopping(patience=3, min_delta=0.001)

for epoch_num in range(EPOCHS):
    model.train()
    total_acc_train = 0
    total_loss_train = 0

    for train_input, train_label in tqdm(
        train_dataloader,
        desc=f"Epoch {epoch_num + 1}/{EPOCHS} [训练]",
    ):
        train_label = train_label.to(device)
        mask = train_input["attention_mask"].to(device)
        input_id = train_input["input_ids"].squeeze(1).to(device)

        outputs = model(
            input_ids=input_id,
            attention_mask=mask,
            labels=train_label,
        )

        batch_loss = outputs.loss
        logits = outputs.logits
        total_loss_train += batch_loss.item() * train_label.size(0)

        acc = (logits.argmax(dim=1) == train_label).sum().item()
        total_acc_train += acc

        model.zero_grad()
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    val_acc, val_precision, val_recall, val_f1, val_loss = evaluate_bert(
        model, val_dataloader, criterion, device
    )
    
    train_loss_avg = total_loss_train / len(train_dataset)
    train_acc_avg = total_acc_train / len(train_dataset)
    
    print(
        f"[Epoch {epoch_num + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss_avg:.4f} | "
        f"Train Acc: {train_acc_avg:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Precision: {val_precision:.4f} | "
        f"Val Recall: {val_recall:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_acc > best_dev_acc:
        best_dev_acc = val_acc
        save_lora_model(model, "best")
        print(f"   🎯 发现更好的模型！验证准确率: {best_dev_acc:.3f}")

    if early_stopping(val_acc):
        print(f"   ⏹️  早停触发，验证准确率连续{early_stopping.patience}个epoch未提升")
        break

train_end_time = time.time()
training_time_sec = train_end_time - train_start_time
training_time_min = training_time_sec / 60

save_lora_model(model, "last")

print("\n✅ 训练完成！")
print(f"   - 最佳验证准确率: {best_dev_acc:.3f}")
print(f"   - 最佳模型已保存: {os.path.join(SAVE_PATH, 'best')}")
print(f"   - 最后模型已保存: {os.path.join(SAVE_PATH, 'last')}")

In [ ]:
"""
目的：在测试集上评估训练好的LoRA模型性能，需要先加载基础模型再加载LoRA适配器

数据流向：
输入：DATA_PATH（数据路径）、tokenizer（分词器）、SAVE_PATH（模型保存路径）、best目录（最佳LoRA适配器）
-> 生成测试数据集 -> 加载基础BERT模型 -> 加载LoRA适配器 -> 评估模型
输出：测试集评估指标（损失、准确率、精确率、召回率、F1分数）

操作步骤：
1. 调用generate_data生成测试集：传入"test"、数据路径、分词器、最大长度、随机种子
2. 创建测试数据加载器：使用DataLoader封装测试集，设置批次大小
3. 导入PeftModel类（用于加载LoRA适配器）
4. 加载基础BERT模型：从MODEL_PATH加载预训练的BERT序列分类模型
5. 加载LoRA适配器：使用PeftModel.from_pretrained从保存路径加载最佳适配器
6. 将模型移动到指定设备（CPU或GPU）
7. 调用evaluate_bert函数在测试集上评估模型，获得所有评估指标
8. 打印实验名称和测试集评估结果（损失、准确率、精确率、召回率、F1分数）
9. 打印训练时间（秒和分钟）
10. 打印最终测试准确率
"""

test_dataset = generate_data("test", DATA_PATH, tokenizer, MAX_LENGTH, RANDOM_SEED)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("\n加载最佳模型进行测试集评估...")
from peft import PeftModel
base_model = BertForSequenceClassification.from_pretrained(MODEL_PATH, config=config)
model = PeftModel.from_pretrained(base_model, os.path.join(SAVE_PATH, "best"))
model = model.to(device)

test_acc, test_precision, test_recall, test_f1, test_loss = evaluate_bert(
    model, test_dataloader, criterion, device
)
print("\n实验名称:"+EXP_NAME)
print("\n测试集评估结果:")
print(f"  - Loss: {test_loss:.3f}")
print(f"  - Accuracy: {test_acc:.3f}")
print(f"  - Precision: {test_precision:.3f}")
print(f"  - Recall: {test_recall:.3f}")
print(f"  - F1 Score: {test_f1:.3f}")
print(f"   - 训练时间: {training_time_sec:.1f} 秒 ({training_time_min:.2f} 分钟)")

print(f"\n🎉 最终测试准确率: {test_acc:.3f}")